# AgentFlow — Training & Evaluation Agent (Bank Marketing Dataset)

This notebook adapts the existing Training & Evaluation Agent experiments to the common Bank Marketing dataset used by the Data Agent, ML Strategy Agent, and Training & Evaluation Agent.

**Task:** Predict whether a customer subscribes to a term deposit (`y`: `yes` / `no`).


The notebook now includes a unified `run_training_evaluation_agent(X, y, agent_input)` entry point. It uses the common 20-model registry and respects the strategy's requested models, metrics, validation type, and fold count.

In [1]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    AdaBoostClassifier, ExtraTreesClassifier, ExtraTreesRegressor,
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.metrics import get_scorer

In [2]:
DATA_PATH = "bank-full.csv"
USE_SAMPLE = True
SAMPLE_SIZE = 5000
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH, sep=";")

print("Original dataset shape:", df.shape)
display(df.head())
print("\nTarget distribution:")
display(df["y"].value_counts())

Original dataset shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no



Target distribution:


y
no     39922
yes     5289
Name: count, dtype: int64

In [3]:
if USE_SAMPLE and len(df) > SAMPLE_SIZE:
    # Stratified sample: preserve the original yes/no target proportions.
    df_sample, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=df["y"]
    )
else:
    df_sample = df.copy()

df_sample = df_sample.reset_index(drop=True)

print("Working dataset shape:", df_sample.shape)
print("\nTarget distribution in working dataset:")
display(df_sample["y"].value_counts(normalize=True).rename("proportion"))

Working dataset shape: (5000, 17)

Target distribution in working dataset:


y
no     0.883
yes    0.117
Name: proportion, dtype: float64

In [4]:
print("Shape:", df_sample.shape)
print("\nData types:")
display(df_sample.dtypes.to_frame("dtype"))

print("\nMissing values by column:")
display(df_sample.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df_sample.duplicated().sum())
print("\nTarget values:", sorted(df_sample["y"].unique()))

Shape: (5000, 17)

Data types:


,dtype
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64



Missing values by column:


,missing_values
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0



Duplicate rows: 0

Target values: ['no', 'yes']


In [5]:
X = df_sample.drop(columns=["y"])
y = df_sample["y"].map({"no": 0, "yes": 1})

numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["number"]).columns.tolist()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

X shape: (5000, 16)
y shape: (5000,)
Numerical columns: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 4000
Testing rows: 1000


In [7]:
agent_input = {
    "problem_type": "classification",
    "recommended_models": [
        "Logistic Regression",
        "Random Forest",
        "Support Vector Machine (SVM)"
    ],
    "evaluation_metrics": [
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ],
    "validation_strategy": {
        "type": "stratified_k_fold",
        "n_splits": 5
    }
}

agent_input

{'problem_type': 'classification',
 'recommended_models': ['Logistic Regression',
  'Random Forest',
  'Support Vector Machine (SVM)'],
 'evaluation_metrics': ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
 'validation_strategy': {'type': 'stratified_k_fold', 'n_splits': 5}}

In [8]:
# -----------------------------------
# Common Model Registry — 10 models per problem type
# -----------------------------------

model_registry = {
    "classification": {
        "Logistic Regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
        "Decision Tree": DecisionTreeClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1
        ),
        "Support Vector Machine (SVM)": SVC(
            probability=True, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Gaussian Naive Bayes": GaussianNB(),
        "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
        "Extra Trees": ExtraTreesClassifier(
            n_estimators=100, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=1
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            random_state=RANDOM_STATE
        )
    },
    "regression": {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(),
        "Lasso Regression": Lasso(max_iter=5000),
        "ElasticNet": ElasticNet(max_iter=5000),
        "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
        "Decision Tree Regressor": DecisionTreeRegressor(random_state=RANDOM_STATE),
        "Random Forest Regressor": RandomForestRegressor(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=1
        ),
        "SVR": SVR(),
        "Gradient Boosting Regressor": GradientBoostingRegressor(
            random_state=RANDOM_STATE
        ),
        "Extra Trees Regressor": ExtraTreesRegressor(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=1
        )
    }
}

def get_models(problem_type, model_names):
    """Select only the requested models from the shared registry."""
    if problem_type not in model_registry:
        raise ValueError(f"Unsupported problem type: {problem_type}")

    available_models = model_registry[problem_type]
    selected_models = {}
    unsupported_models = []

    for model_name in model_names:
        if model_name in available_models:
            selected_models[model_name] = clone(available_models[model_name])
        else:
            unsupported_models.append(model_name)

    return selected_models, unsupported_models

print("Classification models:", len(model_registry["classification"]))
print("Regression models:", len(model_registry["regression"]))

Classification models: 10
Regression models: 10


## Unified Training & Evaluation Agent

This is the single entry point for both classification and regression. The agent receives `X`, `y`, and the strategy dictionary from the ML Strategy Agent. It selects only requested models, validates the strategy, trains the selected models, evaluates only the requested metrics, runs the requested cross-validation strategy, and returns structured results.

The strategy dictionaries in this notebook are examples that simulate the ML Strategy Agent's output; live multi-agent communication is not implemented here.

In [9]:
def run_training_evaluation_agent(X, y, agent_input):
    """
    Run the Training & Evaluation Agent for classification or regression.

    Expected agent_input keys:
      - problem_type: "classification" or "regression"
      - recommended_models: list of model names from model_registry
      - evaluation_metrics: metrics to calculate
      - validation_strategy: {"type": ..., "n_splits": ...}
    """
    problem_type = agent_input["problem_type"]
    recommended_models = agent_input["recommended_models"]
    requested_metrics = agent_input["evaluation_metrics"]
    validation_strategy = agent_input["validation_strategy"]

    if problem_type not in model_registry:
        raise ValueError(f"Unsupported problem type: {problem_type}")

    n_splits = int(validation_strategy.get("n_splits", 5))
    if n_splits < 2:
        raise ValueError("n_splits must be at least 2.")

    strategy_type = validation_strategy.get("type", "").lower()
    if problem_type == "classification":
        if strategy_type not in {"stratified_k_fold", "stratifiedkfold", "stratified"}:
            raise ValueError(
                "Classification requires validation_strategy.type='stratified_k_fold'."
            )
        cv = StratifiedKFold(
            n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE
        )
        allowed_metrics = {"accuracy", "precision", "recall", "f1", "roc_auc"}
        scoring = {
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "roc_auc": "roc_auc"
        }
    else:
        if strategy_type not in {"k_fold", "kfold", "ordinary_k_fold"}:
            raise ValueError(
                "Regression requires validation_strategy.type='k_fold'."
            )
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        allowed_metrics = {"mae", "rmse", "r2"}
        scoring = {
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        }

    unknown_metrics = sorted(set(requested_metrics) - allowed_metrics)
    if unknown_metrics:
        raise ValueError(
            f"Unsupported metrics for {problem_type}: {unknown_metrics}. "
            f"Supported metrics: {sorted(allowed_metrics)}"
        )

    selected_models, unsupported_models = get_models(
        problem_type, recommended_models
    )
    if not selected_models:
        raise ValueError("No supported models were provided.")

    # Split data. Classification uses stratification to preserve class balance.
    if problem_type == "classification":
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=RANDOM_STATE
        )

    # Use the correct preprocessing for each dataset type.
    if problem_type == "classification":
        numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
        categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

        def make_pipeline(model):
            preprocessor = ColumnTransformer(
                transformers=[
                    ("numeric", StandardScaler(), numeric_cols),
                    ("categorical",
                     OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                     categorical_cols)
                ],
                remainder="drop"
            )
            return Pipeline([
                ("preprocessor", preprocessor),
                ("model", clone(model))
            ])
    else:
        def make_pipeline(model):
            return Pipeline([
                ("scaler", StandardScaler()),
                ("model", clone(model))
            ])

    test_results = {}
    cv_results = {}
    trained_models = {}
    training_times = {}

    # Fit each requested model and compute only the metrics requested.
    for model_name, model in selected_models.items():
        pipeline = make_pipeline(model)
        start_time = time.time()
        pipeline.fit(X_train, y_train)
        training_times[model_name] = time.time() - start_time
        trained_models[model_name] = pipeline

        predictions = pipeline.predict(X_test)
        metrics_result = {}

        if problem_type == "classification":
            if "accuracy" in requested_metrics:
                metrics_result["accuracy"] = accuracy_score(y_test, predictions)
            if "precision" in requested_metrics:
                metrics_result["precision"] = precision_score(
                    y_test, predictions, zero_division=0
                )
            if "recall" in requested_metrics:
                metrics_result["recall"] = recall_score(
                    y_test, predictions, zero_division=0
                )
            if "f1" in requested_metrics:
                metrics_result["f1"] = f1_score(
                    y_test, predictions, zero_division=0
                )
            if "roc_auc" in requested_metrics:
                if hasattr(pipeline, "predict_proba"):
                    probabilities = pipeline.predict_proba(X_test)[:, 1]
                    metrics_result["roc_auc"] = roc_auc_score(y_test, probabilities)
                else:
                    raise ValueError(
                        f"{model_name} does not support predict_proba for ROC-AUC."
                    )
        else:
            if "mae" in requested_metrics:
                metrics_result["mae"] = mean_absolute_error(y_test, predictions)
            if "rmse" in requested_metrics:
                metrics_result["rmse"] = mean_squared_error(
                    y_test, predictions
                ) ** 0.5
            if "r2" in requested_metrics:
                metrics_result["r2"] = r2_score(y_test, predictions)

        test_results[model_name] = metrics_result

        # CV computes only the requested scoring metrics.
        requested_scoring = {metric: scoring[metric] for metric in requested_metrics}
        cv_scores = cross_validate(
            make_pipeline(model),
            X_train,
            y_train,
            cv=cv,
            scoring=requested_scoring,
            return_train_score=False,
            n_jobs=1
        )

        cv_model_result = {}
        for metric in requested_metrics:
            scores = cv_scores[f"test_{metric}"]
            if metric in {"mae", "rmse"}:
                scores = -scores  # sklearn loss scorers are negative by convention
            cv_model_result[f"{metric}_mean"] = float(scores.mean())
            cv_model_result[f"{metric}_std"] = float(scores.std())
        cv_results[model_name] = cv_model_result

    agent_output = {
        "status": "success",
        "problem_type": problem_type,
        "dataset_rows": int(len(X)),
        "feature_count": int(X.shape[1]),
        "models_requested": list(recommended_models),
        "models_trained": list(trained_models.keys()),
        "unsupported_models": unsupported_models,
        "evaluation_metrics_requested": list(requested_metrics),
        "validation_strategy": dict(validation_strategy),
        "training_times_seconds": training_times,
        "model_results": {}
    }

    for model_name in trained_models:
        agent_output["model_results"][model_name] = {
            "test_results": test_results[model_name],
            "cross_validation_results": cv_results[model_name]
        }

    return agent_output

In [10]:
selected_models, unsupported_models = get_models(
    agent_input["problem_type"],
    agent_input["recommended_models"]
)

print("Selected models:", list(selected_models.keys()))
print("Unsupported models:", unsupported_models)

if not selected_models:
    raise ValueError("No supported models were provided.")

Selected models: ['Logistic Regression', 'Random Forest', 'Support Vector Machine (SVM)']
Unsupported models: []


In [11]:
def build_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_columns),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns)
        ],
        remainder="drop"
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", clone(model))
    ])

In [12]:
trained_models = {}
training_times = {}

for name, model in selected_models.items():
    print(f"Training {name}...")
    pipeline = build_pipeline(model)

    start_time = time.time()
    pipeline.fit(X_train, y_train)
    elapsed = time.time() - start_time

    trained_models[name] = pipeline
    training_times[name] = elapsed
    print(f"{name} trained in {elapsed:.2f} seconds.")

print("\nTraining times:")
display(pd.DataFrame([
    {"model": name, "training_time_seconds": round(seconds, 3)}
    for name, seconds in training_times.items()
]))

Training Logistic Regression...
Logistic Regression trained in 0.03 seconds.
Training Random Forest...
Random Forest trained in 0.34 seconds.
Training Support Vector Machine (SVM)...
Support Vector Machine (SVM) trained in 1.45 seconds.

Training times:


,model,training_time_seconds
0,Logistic Regression,0.034
1,Random Forest,0.336
2,Support Vector Machine (SVM),1.447


In [13]:
def evaluate_model(model, X_test, y_test, metrics):
    y_pred = model.predict(X_test)
    results = {}

    for metric in metrics:
        if metric == "accuracy":
            results["accuracy"] = accuracy_score(y_test, y_pred)
        elif metric == "precision":
            results["precision"] = precision_score(
                y_test, y_pred, zero_division=0
            )
        elif metric == "recall":
            results["recall"] = recall_score(
                y_test, y_pred, zero_division=0
            )
        elif metric == "f1":
            results["f1"] = f1_score(
                y_test, y_pred, zero_division=0
            )
        elif metric == "roc_auc":
            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(X_test)[:, 1]
                results["roc_auc"] = roc_auc_score(y_test, y_prob)

    return results

evaluation_results = {}
for name, model in trained_models.items():
    evaluation_results[name] = evaluate_model(
        model, X_test, y_test, agent_input["evaluation_metrics"]
    )

test_results_df = pd.DataFrame(evaluation_results).T
display(test_results_df.round(4))

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.849,0.4206,0.7692,0.5438,0.9056
Random Forest,0.899,0.7222,0.2222,0.3399,0.9265
Support Vector Machine (SVM),0.864,0.4541,0.8034,0.5802,0.9241


In [14]:
def perform_cross_validation(model, X, y, validation_strategy, problem_type):
    if problem_type != "classification":
        raise ValueError(f"Unsupported problem type: {problem_type}")

    n_splits = validation_strategy["n_splits"]
    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )
    scoring = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    pipeline = build_pipeline(model)
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1
    )

    return {
        "accuracy_mean": scores["test_accuracy"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "f1_std": scores["test_f1"].std(),
        "roc_auc_mean": scores["test_roc_auc"].mean()
    }

In [15]:
cv_results = {}

for name, model in selected_models.items():
    print(f"Cross-validating {name}...")
    cv_results[name] = perform_cross_validation(
        model,
        X_train,
        y_train,
        agent_input["validation_strategy"],
        agent_input["problem_type"]
    )

cv_results_df = pd.DataFrame(cv_results).T
display(cv_results_df.round(4))

Cross-validating Logistic Regression...
Cross-validating Random Forest...
Cross-validating Support Vector Machine (SVM)...


,accuracy_mean,precision_mean,recall_mean,f1_mean,f1_std,roc_auc_mean
Logistic Regression,0.8350,0.3970,0.7884,0.5277,0.0115,0.8946
Random Forest,0.8960,0.7406,0.1900,0.2977,0.0344,0.9039
Support Vector Machine (SVM),0.8392,0.4078,0.8182,0.5440,0.0239,0.9040


In [16]:
agent_output = {
    "status": "success",
    "problem_type": agent_input["problem_type"],
    "dataset_name": "Bank Marketing",
    "dataset_rows_used": int(len(df_sample)),
    "feature_count": int(X.shape[1]),
    "target_column": "y",
    "models_requested": agent_input["recommended_models"],
    "models_trained": list(trained_models.keys()),
    "unsupported_models": unsupported_models,
    "validation_strategy": agent_input["validation_strategy"],
    "training_times_seconds": training_times,
    "model_results": {}
}

for model_name in trained_models:
    agent_output["model_results"][model_name] = {
        "test_results": evaluation_results[model_name],
        "cross_validation_results": cv_results[model_name]
    }

agent_output

{'status': 'success',
 'problem_type': 'classification',
 'dataset_name': 'Bank Marketing',
 'dataset_rows_used': 5000,
 'feature_count': 16,
 'target_column': 'y',
 'models_requested': ['Logistic Regression',
  'Random Forest',
  'Support Vector Machine (SVM)'],
 'models_trained': ['Logistic Regression',
  'Random Forest',
  'Support Vector Machine (SVM)'],
 'unsupported_models': [],
 'validation_strategy': {'type': 'stratified_k_fold', 'n_splits': 5},
 'training_times_seconds': {'Logistic Regression': 0.03432917594909668,
  'Random Forest': 0.3364992141723633,
  'Support Vector Machine (SVM)': 1.4469106197357178},
 'model_results': {'Logistic Regression': {'test_results': {'accuracy': 0.849,
    'precision': 0.4205607476635514,
    'recall': 0.7692307692307693,
    'f1': 0.5438066465256798,
    'roc_auc': 0.9056247640619102},
   'cross_validation_results': {'accuracy_mean': np.float64(0.835),
    'precision_mean': np.float64(0.39697615824625876),
    'recall_mean': np.float64(0.78837

In [17]:
summary_rows = []

for model_name in trained_models:
    test = evaluation_results[model_name]
    cv = cv_results[model_name]

    summary_rows.append({
        "Model": model_name,
        "Training Time (s)": round(training_times[model_name], 3),
        "Test Accuracy": round(test.get("accuracy", float("nan")), 4),
        "Test Precision": round(test.get("precision", float("nan")), 4),
        "Test Recall": round(test.get("recall", float("nan")), 4),
        "Test F1": round(test.get("f1", float("nan")), 4),
        "Test ROC-AUC": round(test.get("roc_auc", float("nan")), 4),
        "CV Accuracy": round(cv["accuracy_mean"], 4),
        "CV F1": round(cv["f1_mean"], 4),
        "CV F1 Std": round(cv["f1_std"], 4),
        "CV ROC-AUC": round(cv["roc_auc_mean"], 4)
    })

final_results_df = pd.DataFrame(summary_rows)
display(final_results_df)

,Model,Training Time (s),Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,CV Accuracy,CV F1,CV F1 Std,CV ROC-AUC
0,Logistic Regression,0.034,0.849,0.4206,0.7692,0.5438,0.9056,0.8350,0.5277,0.0115,0.8946
1,Random Forest,0.336,0.899,0.7222,0.2222,0.3399,0.9265,0.8960,0.2977,0.0344,0.9039
2,Support Vector Machine (SVM),1.447,0.864,0.4541,0.8034,0.5802,0.9241,0.8392,0.5440,0.0239,0.9040



# Regression Experiment (California Housing Dataset)

This section extends the same Training & Evaluation Agent notebook with a regression workflow. All model classes used by the shared registry are imported in the first code cell, before the registry is created. It uses the built-in California Housing dataset and the same overall design: model registry, dynamic selection, preprocessing, training, test evaluation, five-fold cross-validation, and structured output.

**Regression target:** median house value (in units of $100,000 in this dataset).

In [18]:
# California Housing dataset loader
from sklearn.datasets import fetch_california_housing


In [19]:
housing = fetch_california_housing(as_frame=True)

X_reg = housing.data
y_reg = housing.target

print("Features shape:", X_reg.shape)
print("Target shape:", y_reg.shape)
display(X_reg.head())
display(y_reg.head())

Features shape: (20640, 8)
Target shape: (20640,)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: MedHouseVal, dtype: float64

In [20]:
print("Missing values by feature:")
display(X_reg.isna().sum().to_frame("missing_values"))

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", X_reg_train.shape[0])
print("Testing rows:", X_reg_test.shape[0])

Missing values by feature:


,missing_values
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0


Training rows: 16512
Testing rows: 4128


In [21]:
# -----------------------------------
# Regression Strategy Input
# -----------------------------------

regression_agent_input = {
    "problem_type": "regression",
    "recommended_models": [
        "Linear Regression",
        "Random Forest Regressor",
        "SVR"
    ],
    "evaluation_metrics": ["mae", "rmse", "r2"],
    "validation_strategy": {
        "type": "k_fold",
        "n_splits": 5
    }
}

# Use the same common model registry and selection function
reg_selected_models, reg_unsupported_models = get_models(
    regression_agent_input["problem_type"],
    regression_agent_input["recommended_models"]
)

print("Selected regression models:", list(reg_selected_models.keys()))
print("Unsupported regression models:", reg_unsupported_models)

Selected regression models: ['Linear Regression', 'Random Forest Regressor', 'SVR']
Unsupported regression models: []


In [22]:
def build_regression_pipeline(model):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", clone(model))
    ])

In [23]:
reg_trained_models = {}
reg_training_times = {}

for name, model in reg_selected_models.items():
    print(f"Training {name}...")
    pipeline = build_regression_pipeline(model)

    start_time = time.time()
    pipeline.fit(X_reg_train, y_reg_train)
    elapsed = time.time() - start_time

    reg_trained_models[name] = pipeline
    reg_training_times[name] = elapsed
    print(f"{name} trained in {elapsed:.2f} seconds.")

display(pd.DataFrame([
    {"model": name, "training_time_seconds": round(seconds, 3)}
    for name, seconds in reg_training_times.items()
]))

Training Linear Regression...
Linear Regression trained in 0.01 seconds.
Training Random Forest Regressor...
Random Forest Regressor trained in 11.51 seconds.
Training SVR...
SVR trained in 5.51 seconds.


,model,training_time_seconds
0,Linear Regression,0.011
1,Random Forest Regressor,11.506
2,SVR,5.510


In [24]:
def evaluate_regression_model(model, X_test, y_test, metrics):
    y_pred = model.predict(X_test)
    results = {}

    for metric in metrics:
        if metric == "mae":
            results["mae"] = mean_absolute_error(y_test, y_pred)
        elif metric == "rmse":
            results["rmse"] = mean_squared_error(y_test, y_pred) ** 0.5
        elif metric == "r2":
            results["r2"] = r2_score(y_test, y_pred)

    return results

reg_evaluation_results = {}

for name, model in reg_trained_models.items():
    reg_evaluation_results[name] = evaluate_regression_model(
        model,
        X_reg_test,
        y_reg_test,
        regression_agent_input["evaluation_metrics"]
    )

reg_test_results_df = pd.DataFrame(reg_evaluation_results).T
display(reg_test_results_df.round(4))

,mae,rmse,r2
Linear Regression,0.5332,0.7456,0.5758
Random Forest Regressor,0.3274,0.5051,0.8053
SVR,0.3986,0.5975,0.7276


In [25]:
def perform_regression_cross_validation(
    model,
    X,
    y,
    validation_strategy
):
    n_splits = validation_strategy["n_splits"]
    cv = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2"
    }

    pipeline = build_regression_pipeline(model)
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1
    )

    # sklearn returns negative MAE/RMSE for scorer compatibility;
    # convert them back to positive error values.
    return {
        "mae_mean": -scores["test_mae"].mean(),
        "rmse_mean": -scores["test_rmse"].mean(),
        "r2_mean": scores["test_r2"].mean()
    }

reg_cv_results = {}

for name, model in reg_selected_models.items():
    print(f"Cross-validating {name}...")
    reg_cv_results[name] = perform_regression_cross_validation(
        model,
        X_reg_train,
        y_reg_train,
        regression_agent_input["validation_strategy"]
    )

reg_cv_results_df = pd.DataFrame(reg_cv_results).T
display(reg_cv_results_df.round(4))

Cross-validating Linear Regression...
Cross-validating Random Forest Regressor...
Cross-validating SVR...


,mae_mean,rmse_mean,r2_mean
Linear Regression,0.5291,0.7205,0.6115
Random Forest Regressor,0.3349,0.5109,0.8047
SVR,0.3951,0.5932,0.7365


In [26]:
regression_agent_output = {
    "status": "success",
    "problem_type": "regression",
    "dataset_name": "California Housing",
    "dataset_rows_used": int(len(X_reg)),
    "feature_count": int(X_reg.shape[1]),
    "target_name": "MedHouseVal",
    "models_requested": regression_agent_input["recommended_models"],
    "models_trained": list(reg_trained_models.keys()),
    "unsupported_models": reg_unsupported_models,
    "validation_strategy": regression_agent_input["validation_strategy"],
    "training_times_seconds": reg_training_times,
    "model_results": {}
}

for model_name in reg_trained_models:
    regression_agent_output["model_results"][model_name] = {
        "test_results": reg_evaluation_results[model_name],
        "cross_validation_results": reg_cv_results[model_name]
    }

regression_agent_output

{'status': 'success',
 'problem_type': 'regression',
 'dataset_name': 'California Housing',
 'dataset_rows_used': 20640,
 'feature_count': 8,
 'target_name': 'MedHouseVal',
 'models_requested': ['Linear Regression', 'Random Forest Regressor', 'SVR'],
 'models_trained': ['Linear Regression', 'Random Forest Regressor', 'SVR'],
 'unsupported_models': [],
 'validation_strategy': {'type': 'k_fold', 'n_splits': 5},
 'training_times_seconds': {'Linear Regression': 0.010518312454223633,
  'Random Forest Regressor': 11.505566596984863,
  'SVR': 5.5098021030426025},
 'model_results': {'Linear Regression': {'test_results': {'mae': 0.5332001304956564,
    'rmse': 0.7455813830127764,
    'r2': 0.5757877060324508},
   'cross_validation_results': {'mae_mean': np.float64(0.5290607356851638),
    'rmse_mean': np.float64(0.7205095145054944),
    'r2_mean': np.float64(0.611456745810643)}},
  'Random Forest Regressor': {'test_results': {'mae': 0.3274252027374032,
    'rmse': 0.5051432839771741,
    'r2': 

In [27]:
reg_summary_rows = []

for model_name in reg_trained_models:
    test = reg_evaluation_results[model_name]
    cv = reg_cv_results[model_name]

    reg_summary_rows.append({
        "Model": model_name,
        "Training Time (s)": round(reg_training_times[model_name], 3),
        "Test MAE": round(test["mae"], 4),
        "Test RMSE": round(test["rmse"], 4),
        "Test R²": round(test["r2"], 4),
        "CV MAE": round(cv["mae_mean"], 4),
        "CV RMSE": round(cv["rmse_mean"], 4),
        "CV R²": round(cv["r2_mean"], 4)
    })

reg_final_results_df = pd.DataFrame(reg_summary_rows)
display(reg_final_results_df)

,Model,Training Time (s),Test MAE,Test RMSE,Test R²,CV MAE,CV RMSE,CV R²
0,Linear Regression,0.011,0.5332,0.7456,0.5758,0.5291,0.7205,0.6115
1,Random Forest Regressor,11.506,0.3274,0.5051,0.8053,0.3349,0.5109,0.8047
2,SVR,5.510,0.3986,0.5975,0.7276,0.3951,0.5932,0.7365


## Run the unified agent

These calls demonstrate the same entry point for both problem types. Each strategy input requests three models; the common registry contains all 20 available models.

In [28]:
# Classification: X and y are from the Bank Marketing dataset.
classification_agent_output = run_training_evaluation_agent(
    X, y, agent_input
)

print("Classification models trained:",
      classification_agent_output["models_trained"])
display(pd.DataFrame({
    name: result["test_results"]
    for name, result in classification_agent_output["model_results"].items()
}).T.round(4))

# Regression: X_reg and y_reg are from California Housing.
regression_agent_output_unified = run_training_evaluation_agent(
    X_reg, y_reg, regression_agent_input
)

print("Regression models trained:",
      regression_agent_output_unified["models_trained"])
display(pd.DataFrame({
    name: result["test_results"]
    for name, result in regression_agent_output_unified["model_results"].items()
}).T.round(4))

Classification models trained: ['Logistic Regression', 'Random Forest', 'Support Vector Machine (SVM)']


,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.849,0.4206,0.7692,0.5438,0.9056
Random Forest,0.899,0.7222,0.2222,0.3399,0.9265
Support Vector Machine (SVM),0.864,0.4541,0.8034,0.5802,0.9241


Regression models trained: ['Linear Regression', 'Random Forest Regressor', 'SVR']


,mae,rmse,r2
Linear Regression,0.5332,0.7456,0.5758
Random Forest Regressor,0.3274,0.5051,0.8053
SVR,0.3986,0.5975,0.7276
